In [ ]:
#gpt2_sst2_fine_tuning
from transformers import (
    GPT2Tokenizer,
    GPT2ForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
import numpy as np
import evaluate
import os
from datasets import load_dataset
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Load accuracy metric
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# Step 1: Load SST-2 Dataset from GLUE
print("Loading SST-2 dataset...")
dataset = load_dataset("glue", "sst2")
print(f"Train set size: {len(dataset['train']):,}")
print(f"Validation set size: {len(dataset['validation']):,}")

# Step 2: Initialize the GPT-2 Tokenizer and Assign a Padding Token
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # Use <|endoftext|> as the padding token

# Step 3: Tokenize the Dataset and Add Labels
def tokenize_function_sst2(example):
    tokens = tokenizer(
        example['sentence'],           # SST-2使用sentence字段
        truncation=True,
        padding='max_length',
        max_length=128,                 # SST-2句子较短，128足够
    )
    tokens["labels"] = example['label']
    return tokens

print("Tokenizing SST-2 dataset...")
tokenized_dataset = dataset.map(tokenize_function_sst2, batched=True)
tokenized_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# Select a subset of the dataset (2万条训练，1万条验证)
print("Selecting subsets...")
small_train_dataset = tokenized_dataset["train"].shuffle(seed=42).select(range(20000))
small_eval_dataset = tokenized_dataset["validation"].shuffle(seed=42).select(range(10000))
print(f"Training samples: {len(small_train_dataset)}")
print(f"Evaluation samples: {len(small_eval_dataset)}")

# Step 4: Define the Model for Sequence Classification
model = GPT2ForSequenceClassification.from_pretrained("gpt2", num_labels=2)
model.config.pad_token_id = tokenizer.pad_token_id

# Step 5: Create a Data Collator for Padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

# Step 6: Define the Training Arguments
training_args = TrainingArguments(
    output_dir='./results_gpt2_sst2',
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    learning_rate=2e-5,                 # 更稳定的学习率
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    save_total_limit=2,
)

# Step 7: Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Step 8: Train the Model
print("Starting training on SST-2...")
trainer.train()

# Step 9: Evaluate and Save
print("Evaluating on SST-2 validation set...")
eval_results = trainer.evaluate()
print(f"SST-2 Validation Accuracy: {eval_results['eval_accuracy']:.4f}")

model.save_pretrained('./fine_tuned_gpt2_sst2')
tokenizer.save_pretrained('./fine_tuned_gpt2_sst2')
print("Model saved to ./fine_tuned_gpt2_sst2")

In [ ]:
from transformers import GPT2ForSequenceClassification, GPT2Tokenizer
from datasets import load_dataset
import torch

# 加载训练好的模型和分词器
model_path = './fine_tuned_gpt2_sst2'  # 根据你的保存路径调整
print(f"Loading model from {model_path}...")
model = GPT2ForSequenceClassification.from_pretrained(model_path)
tokenizer = GPT2Tokenizer.from_pretrained(model_path)

# 确保模型在评估模式
model.eval()

# 加载SST-2数据集
print("Loading SST-2 dataset...")
dataset = load_dataset("glue", "sst2")

# 从验证集中抽取一条样本
sample_idx = 42  # 可以改成任意索引
sample = dataset['validation'][sample_idx]
input_text = sample['sentence']
true_label = sample['label']

print(f"\n=== SST-2 单条样本测试 ===")
print(f"样本索引: {sample_idx}")
print(f"输入文本: {input_text}")
print(f"真实标签: {true_label} ({'正面' if true_label == 1 else '负面'})")

# 编码文本
inputs = tokenizer(input_text, return_tensors='pt', truncation=True, max_length=128)

# 推理（不计算梯度）
with torch.no_grad():
    outputs = model(**inputs)

# 获取预测结果
logits = outputs.logits
predicted_class = logits.argmax(dim=-1).item()
probabilities = torch.softmax(logits, dim=-1).squeeze().tolist()

print(f"\n预测结果:")
print(f"  预测类别: {predicted_class} ({'正面' if predicted_class == 1 else '负面'})")
print(f"  预测概率: 负面={probabilities[0]:.4f}, 正面={probabilities[1]:.4f}")
print(f"  {'✅ 预测正确' if predicted_class == true_label else '❌ 预测错误'}")

# 额外测试：试试自定义文本
print(f"\n--- 额外测试自定义文本 ---")
test_texts = [
    "This movie is absolutely fantastic!",
    "Worst film I've ever seen.",
    "It's okay, not great but not terrible."
]

for text in test_texts:
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
    pred = outputs.logits.argmax(dim=-1).item()
    prob = torch.softmax(outputs.logits, dim=-1).squeeze().tolist()
    sentiment = "正面" if pred == 1 else "负面"
    print(f"文本: {text[:30]}... → 预测: {sentiment} (负面概率:{prob[0]:.3f}, 正面概率:{prob[1]:.3f})")

In [ ]:
#gpt2_MNLI
from transformers import (
    GPT2Tokenizer,
    GPT2ForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
import numpy as np
import evaluate
import os
from datasets import load_dataset
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Load accuracy metric
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# Step 1: Load MNLI Dataset from GLUE
print("Loading MNLI dataset...")
dataset = load_dataset("glue", "mnli")
print(f"Train set size: {len(dataset['train']):,}")
print(f"Validation matched size: {len(dataset['validation_matched']):,}")
print(f"Validation mismatched size: {len(dataset['validation_mismatched']):,}")

# Step 2: Initialize the GPT-2 Tokenizer and Assign a Padding Token
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # GPT-2使用右侧填充

# Step 3: Tokenize the Dataset and Add Labels
def tokenize_function_mnli(example):
    # MNLI是句子对任务，需要将前提和假设拼接
    # 使用EOS token作为分隔符
    combined_text = example['premise'] + " " + tokenizer.eos_token + " " + example['hypothesis']
    
    tokens = tokenizer(
        combined_text,
        truncation=True,
        padding='max_length',
        max_length=256,                 # 句子对需要稍长一些
    )
    tokens["labels"] = example['label']
    return tokens

print("Tokenizing MNLI dataset...")
# 只对训练集进行tokenize（验证集会单独处理）
tokenized_train = dataset["train"].map(tokenize_function_mnli, batched=True)
tokenized_train.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# 分别处理匹配集和不匹配集验证集
tokenized_valid_matched = dataset["validation_matched"].map(tokenize_function_mnli, batched=True)
tokenized_valid_matched.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

tokenized_valid_mismatched = dataset["validation_mismatched"].map(tokenize_function_mnli, batched=True)
tokenized_valid_mismatched.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# Select a subset of the dataset (2万条训练，验证集保持完整)
print("Selecting subsets...")
# 确保体裁多样性：按genre分层抽样
train_df = dataset["train"].to_pandas()
genres = train_df['genre'].unique()
samples_per_genre = 20000 // len(genres)  # 10个体裁，每个约2000条

sampled_indices = []
import random
random.seed(42)

for genre in genres:
    genre_indices = train_df[train_df['genre'] == genre].index.tolist()
    if len(genre_indices) > samples_per_genre:
        sampled_indices.extend(random.sample(genre_indices, samples_per_genre))
    else:
        sampled_indices.extend(genre_indices)

sampled_indices = sampled_indices[:20000]  # 确保正好2万条
small_train_dataset = tokenized_train.select(sampled_indices)

# 验证集使用完整的数据（不抽样）
print(f"Training samples: {len(small_train_dataset)}")
print(f"Validation matched samples: {len(tokenized_valid_matched)}")
print(f"Validation mismatched samples: {len(tokenized_valid_mismatched)}")

# Step 4: Define the Model for Sequence Classification
model = GPT2ForSequenceClassification.from_pretrained("gpt2", num_labels=3)  # MNLI是3分类
model.config.pad_token_id = tokenizer.pad_token_id

# Step 5: Create a Data Collator for Padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

# Step 6: Define the Training Arguments
training_args = TrainingArguments(
    output_dir='./results_gpt2_mnli',
    eval_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    save_total_limit=2,
)

# Step 7: Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=tokenized_valid_matched,  # 训练时用匹配集验证
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Step 8: Train the Model
print("Starting training on MNLI...")
trainer.train()

# Step 9: Evaluate on both matched and mismatched sets
print("\nEvaluating on MNLI validation sets...")

# 评估匹配集
eval_matched = trainer.evaluate(eval_dataset=tokenized_valid_matched)
print(f"MNLI Matched Accuracy: {eval_matched['eval_accuracy']:.4f}")

# 评估不匹配集
eval_mismatched = trainer.evaluate(eval_dataset=tokenized_valid_mismatched)
print(f"MNLI Mismatched Accuracy: {eval_mismatched['eval_accuracy']:.4f}")

# Step 10: Save the Fine-Tuned Model
model.save_pretrained('./fine_tuned_gpt2_mnli')
tokenizer.save_pretrained('./fine_tuned_gpt2_mnli')
print("Model saved to ./fine_tuned_gpt2_mnli")

In [ ]:
from transformers import GPT2ForSequenceClassification, GPT2Tokenizer
from datasets import load_dataset
import torch

# 加载训练好的模型和分词器
model_path = './fine_tuned_gpt2_mnli'  # 根据你的保存路径调整
print(f"Loading model from {model_path}...")
model = GPT2ForSequenceClassification.from_pretrained(model_path)
tokenizer = GPT2Tokenizer.from_pretrained(model_path)

# 确保模型在评估模式
model.eval()

# 加载MNLI数据集
print("Loading MNLI dataset...")
dataset = load_dataset("glue", "mnli")

# 从验证集中抽取一条样本（可以选择匹配集或不匹配集）
sample_idx = 100  # 可以改成任意索引
# 可以选择 validation_matched 或 validation_mismatched
sample = dataset['validation_matched'][sample_idx]
premise = sample['premise']
hypothesis = sample['hypothesis']
true_label = sample['label']

# 标签映射
label_map = {0: "蕴含", 1: "中立", 2: "矛盾"}

print(f"\n=== MNLI 单条样本测试 ===")
print(f"样本索引: {sample_idx}")
print(f"前提: {premise}")
print(f"假设: {hypothesis}")
print(f"真实标签: {true_label} ({label_map[true_label]})")

# 注意：MNLI是句子对任务，需要将前提和假设拼接
# 使用和训练时相同的拼接方式：前提 + EOS + 假设
combined_text = premise + " " + tokenizer.eos_token + " " + hypothesis

# 编码文本
inputs = tokenizer(combined_text, return_tensors='pt', truncation=True, max_length=256)

# 推理（不计算梯度）
with torch.no_grad():
    outputs = model(**inputs)

# 获取预测结果
logits = outputs.logits
predicted_class = logits.argmax(dim=-1).item()
probabilities = torch.softmax(logits, dim=-1).squeeze().tolist()

print(f"\n预测结果:")
print(f"  预测类别: {predicted_class} ({label_map[predicted_class]})")
print(f"  预测概率: 蕴含={probabilities[0]:.4f}, 中立={probabilities[1]:.4f}, 矛盾={probabilities[2]:.4f}")
print(f"  {'✅ 预测正确' if predicted_class == true_label else '❌ 预测错误'}")

# 额外测试：从不匹配集也试一条
print(f"\n--- 测试不匹配集样本 ---")
sample_mismatch = dataset['validation_mismatched'][sample_idx]
premise_m = sample_mismatch['premise']
hypothesis_m = sample_mismatch['hypothesis']
true_label_m = sample_mismatch['label']

combined_m = premise_m + " " + tokenizer.eos_token + " " + hypothesis_m
inputs_m = tokenizer(combined_m, return_tensors='pt', truncation=True, max_length=256)

with torch.no_grad():
    outputs_m = model(**inputs_m)

pred_m = outputs_m.logits.argmax(dim=-1).item()
print(f"前提: {premise_m[:50]}...")
print(f"假设: {hypothesis_m[:50]}...")
print(f"真实: {label_map[true_label_m]}, 预测: {label_map[pred_m]}")

# 如果想看更多样本，可以遍历几条
print(f"\n--- 连续测试5条匹配集样本 ---")
correct = 0
for i in range(5):
    sample = dataset['validation_matched'][i]
    premise = sample['premise']
    hypothesis = sample['hypothesis']
    true_label = sample['label']
    
    combined = premise + " " + tokenizer.eos_token + " " + hypothesis
    inputs = tokenizer(combined, return_tensors='pt', truncation=True, max_length=256)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    pred = outputs.logits.argmax(dim=-1).item()
    if pred == true_label:
        correct += 1
    
    print(f"[{i+1}] 前提: {premise[:30]}...")
    print(f"    假设: {hypothesis[:30]}...")
    print(f"    真实: {label_map[true_label]}, 预测: {label_map[pred]}")
    print()

print(f"5条样本中正确: {correct}/5")

In [ ]:
#bert_sst2
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
import numpy as np
import evaluate
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# 1. 加载SST-2数据集
print("="*60)
print("加载 SST-2 数据集...")
print("="*60)
dataset = load_dataset("glue", "sst2")
print(f"训练集: {len(dataset['train']):,} 条")
print(f"验证集: {len(dataset['validation']):,} 条")

# 2. 加载BERT分词器
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")

# 3. 定义分词函数（只截断，不填充）
def tokenize_function_sst2(examples):
    return tokenizer(
        examples["sentence"],  # SST-2使用sentence字段
        truncation=True,
        max_length=128  # SST-2句子较短
    )

# 4. 应用分词函数
print("正在分词处理...")
tokenized_datasets = dataset.map(tokenize_function_sst2, batched=True)

# 5. 抽取子集（2万条训练，1万条验证）
print("正在抽取子集...")
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(20000))
small_eval_dataset = tokenized_datasets["validation"].shuffle(seed=42).select(range(10000))
print(f"训练样本数: {len(small_train_dataset)}")
print(f"验证样本数: {len(small_eval_dataset)}")

# 6. 设置格式
tokenized_datasets.set_format('torch', columns=['input_ids', 'attention_mask'])

# 7. 加载BERT模型（二分类）
print("加载 BERT 模型...")
model = AutoModelForSequenceClassification.from_pretrained(
    "google-bert/bert-base-uncased",
    num_labels=2  # 正面/负面
)

# 8. 创建数据整理器（动态padding）
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 9. 定义评估指标
metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# 10. 训练参数
training_args = TrainingArguments(
    output_dir="./results_bert_sst2",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=5,
    per_device_train_batch_size=32,  # BERT可以更大batch
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    save_total_limit=2,
    logging_dir='./logs_bert_sst2',
    logging_steps=100,
)

# 11. 初始化Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 12. 训练
print("开始训练 BERT on SST-2...")
trainer.train()

# 13. 评估
print("\n评估 BERT on SST-2...")
eval_results = trainer.evaluate()
print(f"SST-2 验证集准确率: {eval_results['eval_accuracy']:.4f}")

# 14. 保存模型
trainer.save_model("./final_bert_sst2")
tokenizer.save_pretrained("./final_bert_sst2")
print("模型已保存到 ./final_bert_sst2")

In [ ]:
#bert_MNLI
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
import numpy as np
import evaluate
import os
import random

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# 1. 加载MNLI数据集
print("="*60)
print("加载 MNLI 数据集...")
print("="*60)
dataset = load_dataset("glue", "mnli")
print(f"训练集: {len(dataset['train']):,} 对")
print(f"验证集(匹配): {len(dataset['validation_matched']):,} 对")
print(f"验证集(不匹配): {len(dataset['validation_mismatched']):,} 对")

# 2. 加载BERT分词器
tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")

# 3. 定义分词函数（BERT原生支持句子对）
def tokenize_function_mnli(examples):
    return tokenizer(
        examples["premise"],      # 前提
        examples["hypothesis"],   # 假设
        truncation=True,
        max_length=256,           # 句子对需要稍长一些
        padding=False,            # 不填充，留给DataCollator
    )

# 4. 应用分词函数
print("正在分词处理...")
# 分别处理训练集和验证集
tokenized_train = dataset["train"].map(tokenize_function_mnli, batched=True)
tokenized_valid_matched = dataset["validation_matched"].map(tokenize_function_mnli, batched=True)
tokenized_valid_mismatched = dataset["validation_mismatched"].map(tokenize_function_mnli, batched=True)

# 5. 设置格式
tokenized_train.set_format('torch', columns=['input_ids', 'attention_mask'])
tokenized_valid_matched.set_format('torch', columns=['input_ids', 'attention_mask'])
tokenized_valid_mismatched.set_format('torch', columns=['input_ids', 'attention_mask'])

# 6. 体裁感知抽样（2万条，覆盖所有体裁）
print("正在进行体裁感知抽样...")
train_df = dataset["train"].to_pandas()
genres = train_df['genre'].unique()
samples_per_genre = 20000 // len(genres)  # 10个体裁，每个约2000条

sampled_indices = []
random.seed(42)

for genre in genres:
    genre_indices = train_df[train_df['genre'] == genre].index.tolist()
    if len(genre_indices) > samples_per_genre:
        sampled_indices.extend(random.sample(genre_indices, samples_per_genre))
    else:
        sampled_indices.extend(genre_indices)

sampled_indices = sampled_indices[:20000]  # 确保正好2万条
small_train_dataset = tokenized_train.select(sampled_indices)

# 验证集保持完整
print(f"训练样本数: {len(small_train_dataset)}")
print(f"验证集(匹配)样本数: {len(tokenized_valid_matched)}")
print(f"验证集(不匹配)样本数: {len(tokenized_valid_mismatched)}")

# 7. 加载BERT模型（三分类）
print("加载 BERT 模型...")
model = AutoModelForSequenceClassification.from_pretrained(
    "google-bert/bert-base-uncased",
    num_labels=3  # 蕴含/中立/矛盾
)

# 8. 创建数据整理器（动态padding）
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 9. 定义评估指标
metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# 10. 训练参数
training_args = TrainingArguments(
    output_dir="./results_bert_mnli",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    save_total_limit=2,
    logging_dir='./logs_bert_mnli',
    logging_steps=100,
)

# 11. 初始化Trainer（用匹配集做验证）
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=tokenized_valid_matched,  # 训练时用匹配集验证
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 12. 训练
print("开始训练 BERT on MNLI...")
trainer.train()

# 13. 在匹配集和不匹配集上评估
print("\n评估 BERT on MNLI...")

# 评估匹配集
eval_matched = trainer.evaluate(eval_dataset=tokenized_valid_matched)
print(f"MNLI 匹配集准确率: {eval_matched['eval_accuracy']:.4f}")

# 评估不匹配集
eval_mismatched = trainer.evaluate(eval_dataset=tokenized_valid_mismatched)
print(f"MNLI 不匹配集准确率: {eval_mismatched['eval_accuracy']:.4f}")

# 14. 保存模型
trainer.save_model("./final_bert_mnli")
tokenizer.save_pretrained("./final_bert_mnli")
print("模型已保存到 ./final_bert_mnli")